In [3]:
import os
import time
import random
from PIL import Image
from google import genai
from google.genai import types
# Load environment variables
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

In [4]:


# Configure built-in SDK automatic retries
retry_options = types.HttpRetryOptions(
    attempts=5,                
    initial_delay=2.0,         
    max_delay=60.0,            
    http_status_codes=[408, 429, 500, 502, 503, 504]
)

client = genai.Client(
    api_key=GOOGLE_API_KEY,
    http_options=types.HttpOptions(retry_options=retry_options)
)

base_folder = "/Users/linuslee/Automation/selected_images"
output_folder = "output_descriptions"
os.makedirs(output_folder, exist_ok=True)

# --- NEW: CLEAN UP OLD FILES TO GUARANTEE ONLY 4 EXIST ---
for old_file in os.listdir(output_folder):
    if old_file.endswith(".txt"):
        os.remove(os.path.join(output_folder, old_file))
print(f"Cleared old files from '{output_folder}' for a fresh start.\n")

# PRODUCT LINK DICTIONARY
PRODUCT_LINKS = {
    "Automatic Nail Clipper": "https://luffcabo.com/products/electric-nail-clipper",
    "Baby Nail File": "https://luffcabo.com/products/baby-nail-file",
    "Baby Sculpting Device": "https://luffcabo.com/products/body-sculpting-machine",
    "Baby Stroller Organizer": "https://luffcabo.com/products/versatile-stroller-organizer",
    "Blue Stroller Cup Holder": "https://luffcabo.com/products/universal-stroller-cup-holder",
    "Breast Pumps": "https://luffcabo.com/products/electric-hands-free-breast-pump",
    "Breathable Baby Carrier": "https://luffcabo.com/products/all-season-baby-carrier",
    "Breathable Baby CarrierM": "https://luffcabo.com/products/luffcabo-mesh-baby-carrier",
    "Detachable Cup Holder": "https://luffcabo.com/products/detachable-stroller-cup-holder",
    "Gray Baby Carrier": "https://luffcabo.com/products/all-season-baby-carrier?variant=41562902167648",
    "Gray stroller cup holder": "https://luffcabo.com/products/universal-stroller-cup-holder",
    "Gray Stroller Organizer": "https://luffcabo.com/products/durable-stroller-organizer",
    "Inflatable Baby Seat": "https://luffcabo.com/products/inflatable-baby-seat",
    "LED Beauty Mask": "https://luffcabo.com/products/led-beauty-mask"  
}

# --- CONFIGURATION POOLS ---
post_1_pool = {"country": "USA", "cities": ["Los Angeles", "Seattle", "New York", "Chicago", "Miami", "San Francisco", "Austin", "Denver"], "language": "English", "day": "Tuesday"}
post_2_pool = {"country": "Germany", "cities": ["Berlin", "Munich", "Frankfurt", "Hamburg", "Cologne", "Stuttgart"], "language": "German", "day": "Thursday"}

post_3_options = [
    {"country": "UK", "cities": ["London", "Manchester", "Edinburgh", "Birmingham", "Bristol"], "language": "English", "day": "Saturday"},
    {"country": "Australia", "cities": ["Sydney", "Melbourne", "Brisbane", "Perth", "Adelaide"], "language": "English", "day": "Saturday"}
]

post_4_options = [
    {"country": "France", "cities": ["Paris", "Marseille", "Lyon", "Toulouse", "Nice", "Bordeaux"], "language": "French", "day": "Sunday"},
    {"country": "Saudi Arabia", "cities": ["Riyadh", "Jeddah", "Mecca", "Medina"], "language": "Arabic", "day": "Sunday"},
    {"country": "United Arab Emirates", "cities": ["Dubai", "Abu Dhabi", "Sharjah"], "language": "Arabic", "day": "Sunday"},
    {"country": "Sweden", "cities": ["Stockholm", "Gothenburg", "Malmö"], "language": "Swedish", "day": "Sunday"},
    {"country": "Netherlands", "cities": ["Amsterdam", "Rotterdam", "The Hague", "Utrecht"], "language": "Dutch", "day": "Sunday"},
    {"country": "Italy", "cities": ["Rome", "Milan", "Naples", "Turin", "Florence"], "language": "Italian", "day": "Sunday"},
    {"country": "Belgium", "cities": ["Brussels", "Antwerp", "Ghent", "Liege"], "language": "French", "day": "Sunday"}
]

image_files = [f for f in os.listdir(base_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
image_files.sort() # Ensure consistent order

# --- NEW: HARD LIMIT TO 4 IMAGES ---
image_files = image_files[:4]
print(f"Selected exactly {len(image_files)} image(s) to process in '{base_folder}'...\n")

for i, image_file in enumerate(image_files):
    img_path = os.path.join(base_folder, image_file)
    
    # Extract product name
    raw_name = os.path.splitext(image_file)[0]
    if raw_name.startswith("modeled_"):
        raw_name = raw_name[len("modeled_"):]
    
    name_parts = raw_name.split('_')
    if name_parts[-1].isdigit():
        name_parts = name_parts[:-1]
    
    base_clean_key = " ".join(name_parts)
    
    # Map link
    if base_clean_key in PRODUCT_LINKS:
        product_link = PRODUCT_LINKS[base_clean_key]
    else:
        url_slug = base_clean_key.lower().replace(" ", "-")
        product_link = f"https://luffcabo.com/products/{url_slug}"

    # DETERMINE CONFIGURATION FOR THIS SPECIFIC IMAGE
    sequence = i % 4 
    
    if sequence == 0:
        country, lang, day = post_1_pool["country"], post_1_pool["language"], post_1_pool["day"]
        city = random.choice(post_1_pool["cities"])
    elif sequence == 1:
        country, lang, day = post_2_pool["country"], post_2_pool["language"], post_2_pool["day"]
        city = random.choice(post_2_pool["cities"])
    elif sequence == 2:
        selection = random.choice(post_3_options)
        country, lang, day = selection["country"], selection["language"], selection["day"]
        city = random.choice(selection["cities"])
    else:
        selection = random.choice(post_4_options)
        country, lang, day = selection["country"], selection["language"], selection["day"]
        city = random.choice(selection["cities"])

    print(f"==========================================")
    print(f"Processing File: {image_file} ({i+1}/{len(image_files)})")
    print(f"Assigned Config: {lang} | {city}, {country} | {day} 7:00 AM")
    
    try:
        img = Image.open(img_path)
    except Exception as e:
        print(f"  - Skipping unreadable image {image_file}: {e}\n")
        continue

    text_prompt = f"""
    Act as a 'Social Media Post Caption' creator and SEO Specialist for LUFFCABO. Your goal is to generate an engaging, attractive, and search-optimized caption and alt text for this single image.

    Analyze the attached image of the '{base_clean_key}'.

    TARGET CONFIGURATION FOR THIS POST:
    - Location: {city}, {country}
    - Schedule: {day}, 7:00 AM Local Time
    - Language: {lang}

    === Behaviors and Rules ===

    1. Caption Composition:
       a) Write a short, engaging paragraph in a 'fun, chill, and attractive' tone that highlights the lifestyle or product appeal.
       b) Incorporate highly Amazon & Google-search-recommended product keywords naturally within the text.
       c) Mandatory Inclusion: You must include the exact phrase (translated organically into {lang}): 'Explore it Now: {product_link}'. The URL must remain exactly as provided.
       d) Hashtags: Conclude with exactly 5 hashtags in this exact format:
          - #LUFFCABO
          - # [1 English core keyword for {base_clean_key}]
          - # [1 Core keyword in {lang}]
          - # [1 Broader lifestyle/product keyword in {lang}]
          - # [1 Broader lifestyle/product keyword in {lang}]
       e) Length: The ENTIRE caption must be strictly under 500 characters.

    2. Alt Text Creation:
       a) Write a concise, descriptive sentence for the alt text field in {lang}.
       b) Seamlessly embed high-ranking Amazon & Google search product keywords.
       c) Focus on clarity for visually impaired users while maintaining SEO benefits.
       d) CRITICAL: NEVER use a period at the end of the alt text.

    3. Language and Style:
       a) Write the entire post (except the English hashtag and URL) strictly in {lang}.
       b) Maintain a consistent brand voice: approachable, trendy, inviting, enthusiastic but relaxed ('chill').
       c) Use emojis where appropriate to enhance the 'fun' vibe.
       d) Keep the content concise and optimized for quick scrolling on social feeds.

    Format your output EXACTLY like this:

    Location Tag: {city}, {country}
    Schedule: {day} 7:00 AM Local Time
    Language: {lang}

    Caption:
    [Your caption here]
    [The 5 Hashtags based on the rule]

    Alt Text:
    [Your alt text here]
    """

    max_manual_retries = 3
    success = False
    
    for attempt in range(1, max_manual_retries + 1):
        try:
            print(f"  - Generating post via Gemini (Attempt {attempt}/{max_manual_retries})...")
            response = client.models.generate_content(
                model='gemini-3.5-flash-lite', # Updated to a standard valid model name for reliability
                contents=[text_prompt, img]
            )
            
            description = response.text
            success = True
            break
            
        except Exception as e:
            print(f"  ⚠️ Attempt {attempt} failed due to error: {e}")
            if attempt < max_manual_retries:
                sleep_time = attempt * 5 
                print(f"  ⏳ High demand detected. Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)
            else:
                print(f"  ❌ All manual retry attempts exhausted for {image_file}.\n")

    if success:
        base_name_without_ext = os.path.splitext(image_file)[0]
        safe_country = country.replace(" ", "")
        txt_filename = f"desc_{base_name_without_ext}_{lang}_{safe_country}.txt"
        txt_output_path = os.path.join(output_folder, txt_filename)
        
        with open(txt_output_path, "w", encoding="utf-8") as text_file:
            text_file.write(description)
            
        print(f"  - Successfully saved to: {txt_output_path}\n")

print("✅ All individual description generations complete!")

Cleared old files from 'output_descriptions' for a fresh start.

Selected exactly 4 image(s) to process in '/Users/linuslee/Automation/selected_images'...

Processing File: modeled_Automatic_Nail_Clipper_2.jpg (1/4)
Assigned Config: English | Miami, USA | Tuesday 7:00 AM
  - Generating post via Gemini (Attempt 1/3)...
  - Successfully saved to: output_descriptions/desc_modeled_Automatic_Nail_Clipper_2_English_USA.txt

Processing File: modeled_Baby_Nail_File_2.jpg (2/4)
Assigned Config: German | Frankfurt, Germany | Thursday 7:00 AM
  - Generating post via Gemini (Attempt 1/3)...
  - Successfully saved to: output_descriptions/desc_modeled_Baby_Nail_File_2_German_Germany.txt

Processing File: modeled_Baby_Sculpting_Device_4.jpg (3/4)
Assigned Config: English | Birmingham, UK | Saturday 7:00 AM
  - Generating post via Gemini (Attempt 1/3)...
  - Successfully saved to: output_descriptions/desc_modeled_Baby_Sculpting_Device_4_English_UK.txt

Processing File: modeled_Baby_Stroller_Organizer_